In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#MLP
import os
import random
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras import layers


FILE_PATH="/content/drive/MyDrive/Mini_Project/"


# ===============================
# REPRODUCIBILITY
# ===============================

SEED=42

os.environ["PYTHONHASHSEED"]=str(SEED)

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ===============================
# LOAD DLQ DATA
# ===============================

df=pd.read_csv(FILE_PATH+"windows_dlq_stream.csv")

print("DLQ logs:",len(df))


# ===============================
# FEATURE ENGINEERING
# ===============================

df["date_and_time"]=pd.to_datetime(df["date_and_time"])

df["hour"]=df["date_and_time"].dt.hour

df["is_night"]=((df["hour"]<6)|(df["hour"]>22)).astype(int)


# ===============================
# FEATURE LIST
# ===============================

features=[

"event_id",
"event_freq",
"burst_count",
"hour",
"is_night"

]

X=df[features]

y = (df["risk_score"] == 2).astype(int)


# ===============================
# TRAIN TEST SPLIT
# ===============================

X_train,X_test,y_train,y_test=train_test_split(

X,
y,
test_size=0.2,
random_state=SEED

)


# ===============================
# SCALING
# ===============================

scaler=RobustScaler()

X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

joblib.dump(scaler,FILE_PATH+"windows_mlp_scaler.pkl")


# ===============================
# BUILD MLP MODEL
# ===============================

model=tf.keras.Sequential([

layers.Input(shape=(X_train.shape[1],)),

layers.Dense(64,activation="relu"),
layers.BatchNormalization(),
layers.Dropout(0.4),

layers.Dense(32,activation="relu"),
layers.Dropout(0.3),

layers.Dense(16,activation="relu"),

layers.Dense(1,activation="sigmoid")

])


model.compile(

optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"]

)

model.summary()


# ===============================
# TRAIN MODEL
# ===============================

history=model.fit(

X_train,
y_train,

epochs=20,
batch_size=1024,

validation_split=0.2

)


# ===============================
# SAVE MODEL
# ===============================

model.save(FILE_PATH+"windows_mlp_model.h5")

print("MLP model saved")


# ===============================
# EVALUATION
# ===============================

y_prob=model.predict(X_test)

y_pred=(y_prob>0.5).astype(int)

print("\nConfusion Matrix")

print(confusion_matrix(y_test,y_pred))

print("\nClassification Report")

print(classification_report(y_test,y_pred))